In [2]:
# ══════════════════════════════════════════════════════════════════
# CELL 0 — Install, upload, merge data, train model
# Must run first. All subsequent cells depend on variables here.
# ══════════════════════════════════════════════════════════════════
!pip install shap pymoo -q

import os, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import shap
warnings.filterwarnings('ignore')
os.makedirs('/content/figures', exist_ok=True)

# ── Upload data files ─────────────────────────────────────────────
from google.colab import files
print("Please upload BOTH files:\n"
      "  1. Questionnaire.xlsx\n"
      "  2. 13_institutions_168h.xlsx")
uploaded = files.upload()

# ── Load questionnaire (Questionnaire.xlsx) ───────────────────────
surv = pd.read_excel('Questionnaire.xlsx')
surv['TSV']  = pd.to_numeric(surv['TSV'], errors='coerce')
surv['date'] = surv['date'].astype(str).str[:10]
surv         = surv.dropna(subset=['TSV']).reset_index(drop=True)
print(f"\n✓ Questionnaire loaded: {len(surv)} valid responses")

# ── Load environmental data (13_institutions_168h.xlsx) ───────────
env = pd.read_excel('13_institutions_168h.xlsx')
env['datetime'] = pd.to_datetime(env['datetime'])
env['date']     = env['datetime'].dt.date.astype(str)
env['hour']     = env['datetime'].dt.hour
print(f"✓ Environmental data loaded: {env.shape[0]} rows, "
      f"{env['institution_id'].nunique()} institutions")

# ── Aggregate questionnaire to institutional time-point means ─────
surv_mean = (surv
             .groupby(['institution_id', 'date', 'hour'])['TSV']
             .agg(TSV_survey='mean', n_resp='count')
             .reset_index())

# ── Merge with environmental data ────────────────────────────────
env_sub = (env[['institution_id', 'date', 'hour', 'district',
                'window_wall_ratio', 'TA_mean', 'RH_mean', 'TG_mean',
                'Total_Radiation_mean', 'Wind_Speed_mean',
                'pm25_mean', 'co2_mean', 'Noise_mean']]
           .rename(columns={
               'window_wall_ratio':    'WWR',
               'Total_Radiation_mean': 'Rad_mean',
               'Wind_Speed_mean':      'WS_mean',
               'pm25_mean':            'PM25_mean',
               'co2_mean':             'CO2_mean',
           }))

matched = (surv_mean
           .merge(env_sub, on=['institution_id', 'date', 'hour'], how='inner')
           .dropna()
           .reset_index(drop=True))

print(f"✓ Merged dataset: n={len(matched)} institutional time-point means")
print(f"  Institutions: {sorted(matched['institution_id'].unique())}")
print(f"  TSV_survey: mean={matched['TSV_survey'].mean():.3f}, "
      f"SD={matched['TSV_survey'].std():.3f}, "
      f"range=[{matched['TSV_survey'].min():.2f}, "
      f"{matched['TSV_survey'].max():.2f}]")

# ── Feature configuration ─────────────────────────────────────────
FEATS = ['TA_mean', 'RH_mean', 'TG_mean', 'Rad_mean',
         'WS_mean', 'PM25_mean', 'CO2_mean', 'WWR', 'Noise_mean']

FEAT_LABELS = {
    'TA_mean':    'Air Temp $T_a$ (°C)',
    'RH_mean':    'Relative Humidity (%)',
    'TG_mean':    'Globe Temp $T_G$ (°C)',
    'Rad_mean':   'Solar Radiation (W/m²)',
    'WS_mean':    'Wind Speed (m/s)',
    'PM25_mean':  '$PM_{2.5}$ (μg/m³)',
    'CO2_mean':   '$CO_2$ (ppm)',
    'WWR':        'Window-Wall Ratio',
    'Noise_mean': 'Noise Level (dB)',
}

PALETTE = {
    'blue':   '#0072B2',
    'red':    '#D55E00',
    'orange': '#E69F00',
    'green':  '#009E73',
    'gray':   '#999999',
    'purple': '#CC79A7',
}

institutions   = sorted(matched['institution_id'].unique())
INST_COLORS    = plt.cm.tab20(np.linspace(0, 1, len(institutions)))

X      = matched[FEATS].values
y      = matched['TSV_survey'].values
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# ── LOO cross-validation ──────────────────────────────────────────
print("\nRunning LOO cross-validation (n=78, ~60 s)...")
loo   = LeaveOneOut()
y_loo = np.zeros(len(y))
for tr_idx, te_idx in loo.split(X_sc):
    m = GradientBoostingRegressor(
            n_estimators=100, max_depth=3,
            learning_rate=0.1, random_state=42)
    m.fit(X_sc[tr_idx], y[tr_idx])
    y_loo[te_idx] = m.predict(X_sc[te_idx])

ss_res    = np.sum((y - y_loo) ** 2)
ss_tot    = np.sum((y - y.mean()) ** 2)
LOO_R2    = 1 - ss_res / ss_tot
LOO_RMSE  = np.sqrt(mean_squared_error(y, y_loo))
LOO_MAE   = mean_absolute_error(y, y_loo)
residuals = y - y_loo
sw_stat, sw_p = stats.shapiro(residuals)

matched['TSV_pred_loo'] = y_loo
matched['residuals']    = residuals

print(f"  LOO R²   = {LOO_R2:.4f}  (paper: 0.3734)")
print(f"  LOO RMSE = {LOO_RMSE:.4f}  (paper: 0.632)")
print(f"  LOO MAE  = {LOO_MAE:.4f}  (paper: 0.487)")
print(f"  Shapiro–Wilk p = {sw_p:.4f}  (paper: 0.782)")

# ── Full-sample model + SHAP ──────────────────────────────────────
print("\nTraining full-sample model and computing SHAP values...")
model_full = GradientBoostingRegressor(
    n_estimators=100, max_depth=3,
    learning_rate=0.1, random_state=42)
model_full.fit(X_sc, y)

explainer      = shap.TreeExplainer(model_full)
shap_values    = explainer.shap_values(X_sc)
mean_abs_shap  = np.abs(shap_values).mean(axis=0)
shap_order     = np.argsort(mean_abs_shap)       # ascending (bar chart order)
shap_order_desc = shap_order[::-1]               # descending

print("\n  SHAP global importance (descending):")
for i in shap_order_desc:
    print(f"    {FEATS[i]:<12}: {mean_abs_shap[i]:.4f}")

# ── Global matplotlib settings ────────────────────────────────────
# ── Install & register Times New Roman font (Colab-safe) ──────────
import subprocess, glob, os, shutil
import matplotlib as mpl
import matplotlib.font_manager as fm

# Step 1: Install msttcorefonts (includes genuine Times New Roman)
subprocess.run(
    ["apt-get", "install", "-y", "-q", "msttcorefonts"],
    check=False, capture_output=True
)

# Step 2: Also copy any TTF/OTF fonts from system to mpl font dir
mpl_font_dir = os.path.join(os.path.dirname(mpl.__file__), 'mpl-data', 'fonts', 'ttf')
for src_dir in ['/usr/share/fonts', '/usr/local/share/fonts', os.path.expanduser('~/.fonts')]:
    for fp in glob.glob(os.path.join(src_dir, '**', '*.tt[fc]'), recursive=True):
        dest = os.path.join(mpl_font_dir, os.path.basename(fp))
        if not os.path.exists(dest):
            try:
                shutil.copy2(fp, dest)
            except Exception:
                pass

# Step 3: Purge all font caches so matplotlib rescans
cache_dir = mpl.get_cachedir()
for fp in glob.glob(os.path.join(cache_dir, '*.json')) +           glob.glob(os.path.join(cache_dir, 'fontlist*.cache')) +           glob.glob(os.path.join(cache_dir, '*.pkl')):
    try:
        os.remove(fp)
    except Exception:
        pass

# Step 4: Rebuild font manager from scratch
fm._load_fontmanager(try_read_cache=False)

# Step 5: Verify Times New Roman is available; fall back to DejaVu Serif
_tnr_found = any(
    'times new roman' in f.name.lower()
    for f in fm.fontManager.ttflist
)
_font_family = 'Times New Roman' if _tnr_found else 'DejaVu Serif'
print(f"  Font selected: {_font_family}  (Times New Roman found: {_tnr_found})")

plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        [_font_family, 'Times New Roman', 'DejaVu Serif',
                          'Liberation Serif', 'Georgia', 'serif'],
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.labelsize':     10,
    'axes.linewidth':      0.8,
    'xtick.labelsize':     9,
    'ytick.labelsize':     9,
    'legend.fontsize':     8.5,
    'mathtext.fontset':   'stix',   # makes math symbols match Times style
    'figure.dpi':          150,     # screen preview
    'savefig.dpi':         600,     # saved at 600 DPI
})

def save_fig(fig, name):
    """Save figure as both PNG and TIFF at 600 DPI."""
    for fmt in ['png', 'tiff']:
        fig.savefig(f'/content/figures/{name}.{fmt}',
                    dpi=600, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  ✓ Saved: /content/figures/{name}.png  &  .tiff")

print("\n✅ Cell 0 complete — all variables ready.")
print("   Proceed to run Fig.1 (framework) then Cells 1–8 in order.")

Please upload BOTH files:
  1. Questionnaire.xlsx
  2. 13_institutions_168h.xlsx


Saving 13_institutions_168h.xlsx to 13_institutions_168h.xlsx

✓ Questionnaire loaded: 234 valid responses
✓ Environmental data loaded: 2184 rows, 13 institutions
✓ Merged dataset: n=78 institutional time-point means
  Institutions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
  TSV_survey: mean=0.346, SD=0.803, range=[-1.33, 2.33]

Running LOO cross-validation (n=78, ~60 s)...
  LOO R²   = 0.3734  (paper: 0.3734)
  LOO RMSE = 0.6316  (paper: 0.632)
  LOO MAE  = 0.4874  (paper: 0.487)
  Shapiro–Wilk p = 0.7816  (paper: 0.782)

Training full-sample model and computing SHAP values...

  SHAP global importance (descending):
    Rad_mean    : 0.3048
    WWR         : 0.1724
    PM25_mean   : 0.1349
    CO2_mean    : 0.1063
    TG_mean     : 0.0921
    RH_mean     : 0.0704
    TA_mean     : 0.0413
    Noise_mean  : 0.0255
    WS_mean     : 0.0125

✅ Cell 0 complet

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — Fig. 2: Indoor Environmental Parameter Distributions
# 4-panel histogram: TA, RH, Solar Radiation, PM2.5
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

panels = [
    ('TA_mean',   'Air Temperature $T_a$ (°C)',
     PALETTE['blue'],   None),
    ('RH_mean',   'Relative Humidity (%)',
     PALETTE['orange'], None),
    ('Rad_mean',  'Solar Radiation (W/m²)',
     PALETTE['red'],    None),
    ('PM25_mean', '$PM_{2.5}$ Concentration (μg/m³)',
     PALETTE['green'],  75.0),   # National Class II standard
]

for ax, (col, xlabel, color, std_line) in zip(axes, panels):
    vals = matched[col].values
    n_bins = 20

    ax.hist(vals, bins=n_bins, color=color, alpha=0.72,
            edgecolor='white', linewidth=0.6)
    ax.axvline(vals.mean(), color='black', linewidth=1.8,
               linestyle='--',
               label=f'Mean = {vals.mean():.1f}')
    ax.axvline(np.median(vals), color=PALETTE['gray'],
               linewidth=1.3, linestyle=':',
               label=f'Median = {np.median(vals):.1f}')
    if std_line is not None:
        ax.axvline(std_line, color='black', linewidth=1.3,
                   linestyle='-.', alpha=0.7,
                   label=f'National standard ({std_line:.0f})')

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Frequency  (n = 78 time-point means)', fontsize=9)
    ax.set_title(
        f'{xlabel.split("(")[0].strip()}\n'
        f'Mean={vals.mean():.1f},  SD={vals.std():.1f},  '
        f'Range=[{vals.min():.1f}, {vals.max():.1f}]',
        fontweight='bold')
    ax.legend(framealpha=0.88, fontsize=8.5)
    ax.grid(axis='y', alpha=0.22, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 2  Indoor Environmental Parameter Distributions\n'
    '(13 institutions × 3 sampling days × 2 time-points;  '
    'n = 78 institutional time-point means)',
    fontsize=11.5, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, 'fig2_env_distributions')

  ✓ Saved: /content/figures/fig2_env_distributions.png  &  .tiff


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — Fig. 3: Survey TSV Distribution and Institution Comparison
# Left: overall histogram (n=234 individual responses)
# Right: boxplot by institution (n=18 each) + mean markers
# ══════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(14, 5.8))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2.2],
                        figure=fig, wspace=0.28)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# ── Panel A: Overall histogram ────────────────────────────────────
tsv_raw   = surv['TSV'].values          # 234 individual responses
bins      = np.arange(-2.5, 3.6, 1.0)
counts, _ = np.histogram(tsv_raw, bins=bins)

ax1.bar(bins[:-1] + 0.5, counts, width=0.82,
        color=PALETTE['blue'], alpha=0.70, edgecolor='white',
        linewidth=0.5)
ax1.axvspan(-0.5, 0.5, color=PALETTE['green'], alpha=0.13,
            label='Neutral zone (−0.5 to +0.5)')
ax1.axvline(tsv_raw.mean(), color=PALETTE['red'], linewidth=2.0,
            linestyle='--',
            label=f'Mean = {tsv_raw.mean():.2f}')

comfort_pct = ((tsv_raw >= -0.5) & (tsv_raw <= 0.5)).mean() * 100
ax1.set_xlabel('Thermal Sensation Vote (TSV)', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_xticks([-2, -1, 0, 1, 2, 3])
ax1.set_xticklabels(
    ['−2\n(Very\nCool)', '−1\n(Cool)', '0\n(Neutral)',
     '+1\n(Sl.\nWarm)',  '+2\n(Warm)',  '+3\n(Hot)'], fontsize=8)
ax1.set_title(
    f'(A) Overall TSV Distribution\n'
    f'n = 234 responses;  '
    f'Neutral zone = {comfort_pct:.1f}%',
    fontweight='bold')
ax1.legend(framealpha=0.88, fontsize=8.5)
ax1.grid(axis='y', alpha=0.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Panel B: Boxplot by institution ───────────────────────────────
inst_means_tsv = matched.groupby('institution_id')['TSV_survey'].mean()
top4           = set(inst_means_tsv.nlargest(4).index)
bp_data        = [surv[surv['institution_id'] == i]['TSV'].values
                  for i in institutions]

bp = ax2.boxplot(
    bp_data, patch_artist=True,
    medianprops=dict(color='black', linewidth=2.2),
    flierprops=dict(marker='o', markersize=3.5,
                    markerfacecolor=PALETTE['gray'],
                    markeredgewidth=0, alpha=0.55),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2))

for patch, inst in zip(bp['boxes'], institutions):
    patch.set_facecolor(PALETTE['red'] if inst in top4
                        else PALETTE['blue'])
    patch.set_alpha(0.65)

# Institution mean markers (diamond)
for j, inst in enumerate(institutions, start=1):
    ax2.plot(j, inst_means_tsv[inst],
             marker='D', color='black', markersize=5.5,
             zorder=5,
             label='Institution mean' if j == 1 else '')

ax2.axhline( 0.5, color=PALETTE['orange'], linewidth=1.3,
             linestyle='--', alpha=0.80,
             label='Comfort threshold ±0.5')
ax2.axhline(-0.5, color=PALETTE['orange'], linewidth=1.3,
             linestyle='--', alpha=0.80)
ax2.axhline(0, color=PALETTE['gray'], linewidth=0.8,
            linestyle=':', alpha=0.45)

ax2.set_xticks(range(1, 14))
ax2.set_xticklabels([f'Inst\n{i}' for i in institutions], fontsize=9)
ax2.set_ylabel('Thermal Sensation Vote (TSV)', fontsize=10)
top4_str = ', '.join(f'Inst {i}' for i in sorted(top4))
ax2.set_title(
    f'(B) TSV by Institution  (n = 18 responses each)\n'
    f'Red = highest-TSV facilities: {top4_str}',
    fontweight='bold')

legend_handles = [
    mpatches.Patch(facecolor=PALETTE['red'],  alpha=0.65,
                   label=f'High-TSV institutions ({top4_str})'),
    mpatches.Patch(facecolor=PALETTE['blue'], alpha=0.65,
                   label='Other institutions'),
    plt.Line2D([0], [0], marker='D', color='black',
               linestyle='None', markersize=5.5,
               label='Institution mean (LOO)'),
    plt.Line2D([0], [0], color=PALETTE['orange'],
               linewidth=1.3, linestyle='--',
               label='Comfort threshold ±0.5'),
]
ax2.legend(handles=legend_handles, fontsize=8.5,
           loc='upper right', framealpha=0.88)
ax2.grid(axis='y', alpha=0.20)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 3  Survey TSV: Overall Distribution and Institution-Level Comparison\n'
    '(Summer 2025; 13 elderly care facilities; Guiyang, China)',
    fontsize=11.5, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'fig3_tsv_distribution')

  ✓ Saved: /content/figures/fig3_tsv_distribution.png  &  .tiff


In [3]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — Fig. 4: LOO-CV Predicted vs. Observed TSV
# Coloured by institution; 1:1 line; ±1.96×RMSE confidence band
# ══════════════════════════════════════════════════════════════════
import builtins # Import builtins to explicitly use built-in min/max
import numpy as np # Import numpy for array operations

fig, ax = plt.subplots(figsize=(6.8, 6.5))

for j, inst in enumerate(institutions):
    mask = matched['institution_id'] == inst
    ax.scatter(
        matched.loc[mask, 'TSV_survey'],
        matched.loc[mask, 'TSV_pred_loo'],
        color=INST_COLORS[j], s=68, alpha=0.82,
        edgecolors='white', linewidths=0.4,
        label=f'Inst {inst}', zorder=3)

# Ensure y and y_loo are treated as NumPy arrays
y_np = np.asarray(y)
y_loo_np = np.asarray(y_loo)

# Use builtins.min and builtins.max to avoid potential shadowing issues
lo = builtins.min(y_np.min(), y_loo_np.min()) - 0.30
hi = builtins.max(y_np.max(), y_loo_np.max()) + 0.30

# 1:1 reference line
ax.plot([lo, hi], [lo, hi], color='black', linewidth=1.6,
        linestyle='--', label='1:1 perfect agreement', zorder=2)

# ±1.96 × RMSE confidence band
x_line = np.linspace(lo, hi, 200)
ax.fill_between(x_line,
                x_line - 1.96 * LOO_RMSE,
                x_line + 1.96 * LOO_RMSE,
                alpha=0.09, color=PALETTE['gray'],
                label=f'±1.96×RMSE (±{1.96*LOO_RMSE:.2f} TSV)')

ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('Observed Institutional Mean TSV (Survey)', fontsize=10)
ax.set_ylabel('Predicted Institutional Mean TSV (LOO-CV)', fontsize=10)
ax.set_title(
    'Fig. 4  LOO Cross-Validation: Predicted vs. Observed Institutional TSV\n'
    'Gradient Boosting Regression;  n = 78 time-point means',
    fontweight='bold')

# Performance annotation box
perf_txt = (f'LOO R²   = {LOO_R2:.4f}\n'
            f'LOO RMSE = {LOO_RMSE:.4f}\n'
            f'LOO MAE  = {LOO_MAE:.4f}\n'
            f'Residual mean = {residuals.mean():.3f}\n'
            f'Shapiro–Wilk p = {sw_p:.3f}')
ax.text(0.04, 0.97, perf_txt,
        transform=ax.transAxes, fontsize=8.8,
        va='top', ha='left', family='monospace',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow',
                  edgecolor=PALETTE['gray'], alpha=0.92))

ax.legend(fontsize=7.5, loc='lower right', ncol=2,
          framealpha=0.88, handlelength=1.2)
ax.grid(alpha=0.20, linewidth=0.5)
ax.set_aspect('equal', adjustable='box')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'fig4_loo_predicted_vs_observed')


  ✓ Saved: /content/figures/fig4_loo_predicted_vs_observed.png  &  .tiff


In [4]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — Fig. 5: Residual Diagnostics (3-panel)
# A: Residuals vs. fitted values
# B: Residual histogram + normal fit
# C: Normal Q-Q plot
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))

# ── Panel A: Residuals vs. fitted ─────────────────────────────────
ax = axes[0]
ax.scatter(y_loo, residuals,
           color=PALETTE['blue'], alpha=0.68, s=50,
           edgecolors='white', linewidths=0.4)
ax.axhline(0, color=PALETTE['red'], linewidth=1.8,
           linestyle='--', label='Zero line')
ax.axhline( 1.96 * LOO_RMSE, color=PALETTE['orange'],
            linewidth=1.1, linestyle=':',
            label=f'+1.96 SD = +{1.96*LOO_RMSE:.2f}')
ax.axhline(-1.96 * LOO_RMSE, color=PALETTE['orange'],
            linewidth=1.1, linestyle=':',
            label=f'−1.96 SD = −{1.96*LOO_RMSE:.2f}')
ax.set_xlabel('Fitted Values (LOO Predicted TSV)', fontsize=10)
ax.set_ylabel('Residuals  (Observed − Predicted)', fontsize=10)
ax.set_title(f'(A) Residuals vs. Fitted\n'
             f'Mean = {residuals.mean():.3f},  '
             f'SD = {residuals.std():.3f}',
             fontweight='bold')
ax.legend(fontsize=8.2)
ax.grid(alpha=0.20); ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel B: Residual histogram + normal fit ──────────────────────
ax = axes[1]
ax.hist(residuals, bins=16, density=True,
        color=PALETTE['blue'], alpha=0.68,
        edgecolor='white', linewidth=0.5)
x_fit = np.linspace(residuals.min() - 0.4, residuals.max() + 0.4, 300)
ax.plot(x_fit,
        stats.norm.pdf(x_fit, residuals.mean(), residuals.std()),
        color=PALETTE['red'], linewidth=2.2,
        label='Normal distribution fit')
ax.set_xlabel('Residuals', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title(f'(B) Residual Distribution\n'
             f'Shapiro–Wilk:  W = {sw_stat:.3f},  p = {sw_p:.3f}',
             fontweight='bold')
ax.legend(fontsize=8.2)
ax.grid(alpha=0.20); ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel C: Normal Q-Q plot ──────────────────────────────────────
ax = axes[2]
(osm, osr), (slope, intercept, r_qq) = stats.probplot(residuals,
                                                        dist='norm')
ax.scatter(osm, osr, color=PALETTE['blue'], s=40, alpha=0.75,
           edgecolors='white', linewidths=0.3)
x_qq = np.array([osm.min(), osm.max()])
ax.plot(x_qq, slope * x_qq + intercept,
        color=PALETTE['red'], linewidth=2.0, linestyle='--',
        label=f'Reference line  (r = {r_qq:.3f})')
ax.set_xlabel('Theoretical Quantiles (Normal)', fontsize=10)
ax.set_ylabel('Sample Quantiles (Residuals)', fontsize=10)
ax.set_title(f'(C) Normal Q–Q Plot\n'
             f'Pearson r = {r_qq:.3f}',
             fontweight='bold')
ax.legend(fontsize=8.2)
ax.grid(alpha=0.20); ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 5  Residual Diagnostics — Gradient Boosting LOO Cross-Validation\n'
    '(n = 78 institutional time-point means)',
    fontsize=11.5, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'fig5_residual_diagnostics')

  ✓ Saved: /content/figures/fig5_residual_diagnostics.png  &  .tiff


In [10]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — Fig. 6: SHAP Global Bar Chart
#           Fig. 7: SHAP Beeswarm Dot Plot
# ══════════════════════════════════════════════════════════════════
feat_labels_list = [FEAT_LABELS[f] for f in FEATS]

# ── Fig. 6: SHAP global feature importance bar chart ─────────────
fig, ax = plt.subplots(figsize=(8.5, 5.8))

bar_colors = [
    PALETTE['red'] if mean_abs_shap[i] >= sorted(mean_abs_shap)[-2]
    else PALETTE['blue']
    for i in shap_order
]
ax.barh(range(len(FEATS)), mean_abs_shap[shap_order],
        color=bar_colors, alpha=0.80,
        edgecolor='white', height=0.68)

ax.set_yticks(range(len(FEATS)))
ax.set_yticklabels(
    [feat_labels_list[i] for i in shap_order], fontsize=10)
ax.set_xlabel('Mean |SHAP Value| — Global Feature Importance',
              fontsize=10)
ax.set_title(
    'Fig. 6  SHAP Global Feature Importance\n'
    '(Full-sample model; n = 78;  '
    'top-2 predictors highlighted in red)',
    fontweight='bold')

# Value labels
for i, (idx, val) in enumerate(
        zip(shap_order, mean_abs_shap[shap_order])):
    ax.text(val + 0.003, i, f'{val:.3f}',
            va='center', ha='left', fontsize=8.8)

# Cumulative share annotation
cum_top2 = (mean_abs_shap[shap_order_desc[:2]].sum()
            / mean_abs_shap.sum() * 100)
ax.text(0.38, 0.02,
        f'Top-2 features account for\n{cum_top2:.1f}% of total importance',
        transform=ax.transAxes, fontsize=8.5, ha='left', va='bottom',
        bbox=dict(boxstyle='round', facecolor='lightyellow',
                  edgecolor=PALETTE['gray'], alpha=0.92))

ax.legend(handles=[
    mpatches.Patch(fc=PALETTE['red'],  alpha=0.80,
                   label='Top-2 dominant predictors'),
    mpatches.Patch(fc=PALETTE['blue'], alpha=0.80,
                   label='Other features'),
], loc='lower right', fontsize=8.8)
ax.set_xlim(0, mean_abs_shap.max() * 1.22)
ax.grid(axis='x', alpha=0.22, linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'fig6_shap_bar')

# ── Fig. 7: SHAP Beeswarm dot plot ───────────────────────────────
fig, ax = plt.subplots(figsize=(10.5, 6.2))

for i, feat_idx in enumerate(shap_order):
    feat_vals  = X_sc[:, feat_idx]
    shap_vals  = shap_values[:, feat_idx]
    norm_vals  = ((feat_vals - feat_vals.min())
                  / (feat_vals.max() - feat_vals.min() + 1e-9))
    rng        = np.random.RandomState(feat_idx + 7)
    jitter     = rng.uniform(-0.22, 0.22, size=len(shap_vals))
    sc = ax.scatter(shap_vals, i + jitter,
                    c=norm_vals, cmap='RdBu_r',
                    vmin=0, vmax=1,
                    alpha=0.68, s=30, zorder=3)

ax.axvline(0, color='black', linewidth=1.0, zorder=2)
ax.set_yticks(range(len(FEATS)))
ax.set_yticklabels(
    [feat_labels_list[i] for i in shap_order], fontsize=10)
ax.set_xlabel(
    'SHAP Value  (Impact on Predicted Institutional Mean TSV)',
    fontsize=10)
ax.set_title(
    'Fig. 7  SHAP Dot Plot (Beeswarm)\n'
    '(Each point = 1 institutional time-point;  '
    'colour = normalised feature value)',
    fontweight='bold')

cbar = plt.colorbar(sc, ax=ax, fraction=0.024, pad=0.02)
cbar.set_label('Feature Value\n(Blue = Low  |  Red = High)',
               fontsize=8.5)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(['Low', 'Mid', 'High'], fontsize=8)

ax.grid(axis='x', alpha=0.20, linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'fig7_shap_beeswarm')

  ✓ Saved: /content/figures/fig6_shap_bar.png  &  .tiff


  ✓ Saved: /content/figures/fig7_shap_beeswarm.png  &  .tiff


In [6]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — Fig. 8: SHAP Dependence Plot — Solar Radiation × WWR
# Left : scatter SHAP(Rad) vs Rad, coloured by WWR + LOWESS trend
# Right: boxplot of SHAP(Rad) by WWR tertile + ANOVA test
# ══════════════════════════════════════════════════════════════════
rad_idx  = FEATS.index('Rad_mean')
wwr_idx  = FEATS.index('WWR')
rad_orig = scaler.inverse_transform(X_sc)[:, rad_idx]
wwr_orig = matched['WWR'].values
wwr_p33  = np.percentile(wwr_orig, 33)
wwr_p67  = np.percentile(wwr_orig, 67)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

# ── Panel A: SHAP(Rad) vs Solar Radiation, coloured by WWR ───────
ax = axes[0]
sc = ax.scatter(
    rad_orig, shap_values[:, rad_idx],
    c=wwr_orig, cmap='RdYlBu_r',
    vmin=wwr_orig.min(), vmax=wwr_orig.max(),
    alpha=0.82, s=60,
    edgecolors='white', linewidths=0.3, zorder=3)

# LOWESS smoothed trend
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    trend = lowess(shap_values[:, rad_idx], rad_orig,
                   frac=0.40, it=2, return_sorted=True)
    ax.plot(trend[:, 0], trend[:, 1],
            color='black', linewidth=2.2, zorder=4,
            label='LOWESS trend')
except ImportError:
    z  = np.polyfit(rad_orig, shap_values[:, rad_idx], 2)
    xs = np.linspace(rad_orig.min(), rad_orig.max(), 300)
    ax.plot(xs, np.poly1d(z)(xs),
            color='black', linewidth=2.2, zorder=4,
            label='Polynomial trend (degree 2)')

ax.axvline(300, color=PALETTE['red'], linewidth=1.4,
           linestyle='--', alpha=0.85,
           label='~300 W/m²  inflection point')
ax.axhline(0, color=PALETTE['gray'], linewidth=0.8,
           linestyle=':', alpha=0.55)

cbar = plt.colorbar(sc, ax=ax, fraction=0.045, pad=0.02)
cbar.set_label('Window-Wall Ratio (WWR)', fontsize=8.8)
ax.set_xlabel('Solar Radiation (W/m²)', fontsize=10)
ax.set_ylabel('SHAP Value for Solar Radiation', fontsize=10)
ax.set_title(
    '(A) SHAP(Solar Radiation) vs. Solar Radiation\n'
    'Coloured by WWR — higher WWR amplifies SHAP magnitude',
    fontweight='bold')
ax.legend(fontsize=8.5)
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── Panel B: SHAP(Rad) by WWR tertile — boxplot + ANOVA ──────────
ax = axes[1]
groups = {
    f'Low WWR\n(≤ {wwr_p33:.2f})':
        shap_values[wwr_orig <= wwr_p33, rad_idx],
    f'Mid WWR\n({wwr_p33:.2f} – {wwr_p67:.2f})':
        shap_values[(wwr_orig > wwr_p33) & (wwr_orig <= wwr_p67), rad_idx],
    f'High WWR\n(> {wwr_p67:.2f})':
        shap_values[wwr_orig > wwr_p67, rad_idx],
}
grp_colors = [PALETTE['blue'], PALETTE['orange'], PALETTE['red']]

bps = ax.boxplot(
    list(groups.values()), patch_artist=True, positions=[1, 2, 3],
    medianprops=dict(color='black', linewidth=2.2),
    flierprops=dict(marker='o', markersize=4.0, alpha=0.55),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2))

for patch, col in zip(bps['boxes'], grp_colors):
    patch.set_facecolor(col); patch.set_alpha(0.68)

for pos, vals in zip([1, 2, 3], groups.values()):
    ax.plot(pos, np.mean(vals), marker='D',
            color='black', markersize=6.5, zorder=5)

f_stat, p_anova = stats.f_oneway(*list(groups.values()))
ax.text(0.97, 0.97,
        f'One-way ANOVA\nF = {f_stat:.2f},  p = {p_anova:.3f}',
        transform=ax.transAxes, fontsize=8.8,
        va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow',
                  edgecolor=PALETTE['gray'], alpha=0.92))

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(list(groups.keys()), fontsize=9.2)
ax.set_ylabel('SHAP Value for Solar Radiation', fontsize=10)
ax.set_title(
    '(B) SHAP(Solar Radiation) Stratified by WWR Tertile\n'
    '(Diamond = group mean;  '
    'higher WWR → greater solar radiation impact)',
    fontweight='bold')
ax.axhline(0, color=PALETTE['gray'], linewidth=0.8,
           linestyle=':', alpha=0.55)
ax.grid(axis='y', alpha=0.20)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 8  SHAP Dependence Plot: Solar Radiation × Window-Wall Ratio Interaction\n'
    '(Urban microclimate → building envelope → indoor thermal sensation pathway)',
    fontsize=11.5, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'fig8_shap_dependence')

  ✓ Saved: /content/figures/fig8_shap_dependence.png  &  .tiff


In [11]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — Fig. 9: NSGA-II Multi-Objective Pareto Front
# Obj 1: P(TSV > 0.5)  via surrogate GB model
# Obj 2: Energy (kWh/m²)  via parametric model from [20]
# Decision vars: WWR ∈ [0.10, 0.73];  ACH ∈ [0.5, 3.0 h⁻¹]
# ══════════════════════════════════════════════════════════════════
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize as pymoo_minimize
from pymoo.termination import get_termination

class ThermalEnergyProblem(Problem):
    """
    f1 = P(TSV_pred > 0.5) = sigmoid(2 × (tsv_mean − 0.5))
         Surrogate: gradient boosting model with median env. conditions,
         varying only WWR.
    f2 = 0.70 × (1 + 0.15×WWR) × (1 + 0.08×ACH)  [kWh/m²]
         Parametric energy model (Zhang et al., Build. Simul. 2025).
    """
    def __init__(self):
        super().__init__(
            n_var=2, n_obj=2, n_ieq_constr=0,
            xl=np.array([0.10, 0.5]),
            xu=np.array([0.73, 3.0]))
        self.med_env = matched[FEATS].median().values.copy()
        self.wwr_i   = FEATS.index('WWR')

    def _evaluate(self, X_dec, out, *args, **kwargs):
        f1 = np.zeros(len(X_dec))
        f2 = np.zeros(len(X_dec))
        for k, (wwr, ach) in enumerate(X_dec):
            feat = self.med_env.copy()
            feat[self.wwr_i] = wwr
            tsv  = float(model_full.predict(
                         scaler.transform(feat.reshape(1, -1)))[0])
            f1[k] = 1.0 / (1.0 + np.exp(-2.0 * (tsv - 0.5)))
            f2[k] = 0.70 * (1 + 0.15 * wwr) * (1 + 0.08 * ach)
        out['F'] = np.column_stack([f1, f2])

print("Running NSGA-II  (pop=200, gen=500) — approximately 2–3 min...")
res = pymoo_minimize(
    ThermalEnergyProblem(),
    NSGA2(pop_size=200),
    get_termination('n_gen', 500),
    seed=42, verbose=False)

pf, pv = res.F, res.X
print(f"  Pareto-front solutions: {len(pf)}")

# Knee-point: min Euclidean distance to normalised ideal point
pf_norm  = ((pf - pf.min(0))
            / (pf.max(0) - pf.min(0) + 1e-9))
knee_idx = np.argmin(np.linalg.norm(pf_norm, axis=1))
knee_f   = pf[knee_idx]
knee_v   = pv[knee_idx]
print(f"  Knee-point:  WWR={knee_v[0]:.3f},  ACH={knee_v[1]:.3f}")
print(f"               f1={knee_f[0]:.4f},  f2={knee_f[1]:.4f} kWh/m²")

# Observed baseline (mean WWR across all institutions)
base_wwr  = matched['WWR'].mean()
feat_base = matched[FEATS].median().values.copy()
feat_base[FEATS.index('WWR')] = base_wwr
tsv_base  = float(model_full.predict(
                  scaler.transform(feat_base.reshape(1, -1)))[0])
f1_base   = 1.0 / (1.0 + np.exp(-2.0 * (tsv_base - 0.5)))
f2_base   = 0.70 * (1 + 0.15 * base_wwr) * (1 + 0.08 * 0.5)

# ── Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.5, 6.0))

sort_idx = np.argsort(pf[:, 0])
ax.plot(pf[sort_idx, 0], pf[sort_idx, 1],
        color=PALETTE['gray'], linewidth=0.9, alpha=0.45, zorder=1)

sc = ax.scatter(pf[:, 0], pf[:, 1],
                c=pv[:, 0], cmap='RdYlBu_r',
                vmin=pv[:, 0].min(), vmax=pv[:, 0].max(),
                s=30, alpha=0.78, edgecolors='none', zorder=2)

# Knee-point star marker
ax.scatter(knee_f[0], knee_f[1],
           color='black', s=180, marker='*', zorder=5,
           label=f'Knee-point\n'
                 f'WWR = {knee_v[0]:.2f},  ACH = {knee_v[1]:.2f}\n'
                 f'f₁ = {knee_f[0]:.3f},  '
                 f'f₂ = {knee_f[1]:.3f} kWh/m²')

# Baseline square marker
ax.scatter(f1_base, f2_base,
           color=PALETTE['red'], s=120, marker='s', zorder=5,
           label=f'Observed baseline (mean WWR = {base_wwr:.2f})\n'
                 f'f₁ = {f1_base:.3f},  f₂ = {f2_base:.3f} kWh/m²')

# Improvement arrow
ax.annotate('', xy=(knee_f[0], knee_f[1]),
            xytext=(f1_base, f2_base),
            arrowprops=dict(arrowstyle='->',
                            color='black', lw=1.8,
                            mutation_scale=14), zorder=4)
d1 = (f1_base - knee_f[0]) / f1_base * 100
d2 = (f2_base - knee_f[1]) / f2_base * 100
ax.text((knee_f[0] + f1_base) / 2 + 0.01,
        (knee_f[1] + f2_base) / 2 - 0.003,
        f'−{d1:.0f}% discomfort\n−{d2:.0f}% energy',
        fontsize=9.0, color='black',
        bbox=dict(boxstyle='round', facecolor='white',
                  edgecolor=PALETTE['gray'], alpha=0.90))

cbar = plt.colorbar(sc, ax=ax, fraction=0.038, pad=0.02)
cbar.set_label('WWR (Decision Variable)', fontsize=8.8)
ax.set_xlabel(
    'Objective 1: Thermal Discomfort  P(TSV > 0.5)', fontsize=10)
ax.set_ylabel(
    'Objective 2: Energy Consumption (kWh/m²)', fontsize=10)
ax.set_title(
    'Fig. 9  NSGA-II Multi-Objective Pareto Front\n'
    '(Decision variables: WWR ∈ [0.10, 0.73];  '
    'ACH ∈ [0.5, 3.0 h⁻¹])',
    fontweight='bold')
ax.legend(fontsize=8.5, loc='lower right', framealpha=0.92)
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'fig9_pareto_front')

Running NSGA-II  (pop=200, gen=500) — approximately 2–3 min...


  Pareto-front solutions: 31
  Knee-point:  WWR=0.390,  ACH=0.500
               f1=0.2357,  f2=0.7706 kWh/m²


  ✓ Saved: /content/figures/fig9_pareto_front.png  &  .tiff


In [12]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — Fig. 10: Learning Curve  +  Pack & Download all figures
# ══════════════════════════════════════════════════════════════════
print("Computing learning curve (LOO-CV, ~2 min)...")

# Ensure y is the correct array-like format to avoid InvalidParameterError
y_array = matched['TSV_survey'].values
train_fracs = np.linspace(0.30, 1.0, 10)

tr_sizes, tr_scores, val_scores = learning_curve(
    GradientBoostingRegressor(
        n_estimators=100, max_depth=3,
        learning_rate=0.1, random_state=42),
    X_sc, y_array,
    train_sizes=train_fracs,
    cv=LeaveOneOut(),
    scoring='neg_mean_squared_error',
    n_jobs=-1)

tr_mse  = -tr_scores
val_mse = -val_scores

fig, ax = plt.subplots(figsize=(7.8, 5.5))

ax.plot(tr_sizes, tr_mse.mean(axis=1),
        color=PALETTE['blue'], marker='o', markersize=6.5,
        linewidth=1.8, label='Training MSE', zorder=3)
ax.fill_between(
    tr_sizes,
    tr_mse.mean(axis=1) - tr_mse.std(axis=1),
    tr_mse.mean(axis=1) + tr_mse.std(axis=1),
    alpha=0.14, color=PALETTE['blue'])

ax.plot(tr_sizes, val_mse.mean(axis=1),
        color=PALETTE['orange'], marker='s', markersize=6.5,
        linewidth=1.8, label='LOO Validation MSE', zorder=3)
ax.fill_between(
    tr_sizes,
    val_mse.mean(axis=1) - val_mse.std(axis=1),
    val_mse.mean(axis=1) + val_mse.std(axis=1),
    alpha=0.14, color=PALETTE['orange'])

final_val_mse = val_mse.mean(axis=1)[-1]
ax.axhline(LOO_RMSE ** 2, color=PALETTE['red'],
           linewidth=1.6, linestyle='--',
           label=f'Final LOO MSE = {LOO_RMSE**2:.4f}  '
                 f'(RMSE = {LOO_RMSE:.4f})')

ax.set_xlabel('Training Set Size (samples)', fontsize=10)
ax.set_ylabel('Mean Squared Error (MSE)', fontsize=10)
ax.set_title(
    'Fig. 10  Learning Curve — Gradient Boosting Regression\n'
    '(LOO Cross-Validation;  shaded bands = ±1 SD)',
    fontweight='bold')
ax.legend(fontsize=9.0, framealpha=0.92)
ax.grid(alpha=0.22)
ax.set_xlim(tr_sizes.min() - 1, tr_sizes.max() + 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'fig10_learning_curve')

# ── Pack and download all figures ─────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('/content/paper_figures_600dpi', 'zip',
                    '/content/figures')
files.download('/content/paper_figures_600dpi.zip')

print("\n✅ All figures packaged and downloaded.")
print("\nContents:")
for f in sorted(os.listdir('/content/figures')):
    kb = os.path.getsize(f'/content/figures/{f}') / 1024
    print(f"  {f:<50}  {kb:6.0f} KB")

print("\nFigure → Paper section mapping:")
mapping = [
    ('fig1_framework',              'Framework diagram       (Fig. 1)'),
    ('fig2_env_distributions',      'Section 4.1             (Fig. 2)'),
    ('fig3_tsv_distribution',       'Section 4.1             (Fig. 3)'),
    ('fig4_loo_predicted_vs_observed','Section 4.2            (Fig. 4)'),
    ('fig5_residual_diagnostics',   'Section 4.2             (Fig. 5)'),
    ('fig6_shap_bar',               'Section 4.3             (Fig. 6)'),
    ('fig7_shap_beeswarm',          'Section 4.3             (Fig. 7)'),
    ('fig8_shap_dependence',        'Section 4.3             (Fig. 8)'),
    ('fig9_pareto_front',           'Section 4.4             (Fig. 9)'),
    ('fig10_learning_curve',        'Section 4.2             (Fig. 10)'),
]
for name, section in mapping:
    print(f"  {name:<45}  →  {section}")

Computing learning curve (LOO-CV, ~2 min)...


  ✓ Saved: /content/figures/fig10_learning_curve.png  &  .tiff


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All figures packaged and downloaded.

Contents:
  fig10_learning_curve.png                               412 KB
  fig10_learning_curve.tiff                            58394 KB
  fig1_final_final.png                                  1255 KB
  fig4_loo_predicted_vs_observed.png                     684 KB
  fig4_loo_predicted_vs_observed.tiff                  58394 KB
  fig5_residual_diagnostics.png                          956 KB
  fig5_residual_diagnostics.tiff                       96606 KB
  fig6_shap_bar.png                                      478 KB
  fig6_shap_bar.tiff                                   67174 KB
  fig7_shap_beeswarm.png                                1515 KB
  fig7_shap_beeswarm.tiff                              89034 KB
  fig8_shap_dependence.png                               903 KB
  fig8_shap_dependence.tiff                           117762 KB
  fig9_pareto_front.png                                  584 KB
  fig9_pareto_front.tiff                             

In [9]:
"""
Fig. 1 — FINAL FINAL VERSION
- Left 01-06 number size reduced to 12pt (most compact and unobtrusive)
- All other elements perfectly aligned, no overlaps
"""
import subprocess, os, glob, warnings
warnings.filterwarnings('ignore')
subprocess.run(['apt-get','install','-y','msttcorefonts','-qq'], capture_output=True)
import matplotlib.font_manager as fm
for f in glob.glob(os.path.join(matplotlib.get_cachedir(),'*.json')):
    os.remove(f)
try:
    fm._load_fontmanager(try_read_cache=False)
except Exception:
    pass
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon
import numpy as np, os
os.makedirs('/content/figures', exist_ok=True)
names = [f.name for f in fm.fontManager.ttflist]
FONT = 'Times New Roman' if 'Times New Roman' in names else 'DejaVu Serif'

C={'z1f':'#EBF5FB','z1e':'#185FA5','z1d':'#0C447C',
   'z2f':'#EEEDFE','z2e':'#534AB7','z2d':'#3C3489',
   'z3f':'#E1F5EE','z3e':'#1D9E75','z3d':'#085041',
   'z4f':'#EEEDFE','z4e':'#534AB7','z4d':'#3C3489',
   'z5f':'#FEF9E7','z5e':'#BA7517','z5d':'#412402',
   'z6f':'#FAECE7','z6e':'#993C1D','z6d':'#4A1B0C',
   'z7f':'#B03A2E','c1':'#185FA5','c2':'#534AB7','c3':'#7F77DD','c4':'#B4B2A9',
   's1':'#185FA5','s2':'#534AB7','s3':'#7F77DD','s4':'#888780',
   'wh':'#FFFFFF','dk':'#1C2833','lt':'#CCCCCC','mu':'#5D6D7E','fa':'#D5D8DC'}
DPI=600; FW,FH=10.0,6.2
fig=plt.figure(figsize=(FW,FH),dpi=DPI,facecolor='white')
ax=fig.add_axes([0,0,1,1]); ax.set_xlim(0,FW); ax.set_ylim(0,FH); ax.axis('off')
CX=3.50
ZG={1:(5.26,6.00,2.50,2.90),2:(4.58,5.26,2.00,2.50),3:(3.76,4.58,1.70,2.00),
    4:(3.08,3.76,1.40,1.70),5:(1.58,3.08,2.10,1.40),6:(0.76,1.58,2.90,2.10)}
OUT_YB=0.20; OUT_H=0.56; RP_X=7.30; RP_W=2.55; RP_YT=6.00

def hw_at(y,zn):
    yb,yt,hwb,hwt=ZG[zn]; t=(y-yb)/(yt-yb); return hwb+(hwt-hwb)*t
def rbox(ax,x,y,w,h,fc,ec='none',lw=0.5,r=0.06,z=3):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle=f'round,pad=0.01,rounding_size={r}',fc=fc,ec=ec,lw=lw,zorder=z))
def pill(ax,xc,yc,w,h,fc,ec='none',lw=0,z=5):
    r=min(h/2*0.90,0.10)
    ax.add_patch(FancyBboxPatch((xc-w/2,yc-h/2),w,h,boxstyle=f'round,pad=0.005,rounding_size={r}',fc=fc,ec=ec,lw=lw,zorder=z))
def T(ax,x,y,s,col,fs,ha='center',va='center',bold=False,italic=False,z=8):
    ax.text(x,y,s,color=col,fontsize=fs,ha=ha,va=va,zorder=z,fontweight='bold' if bold else 'normal',style='italic' if italic else 'normal',fontfamily=FONT)
def hl(ax,y,x0,x1,col=None,lw=0.4): ax.plot([x0,x1],[y,y],color=col or C['lt'],lw=lw,zorder=1)

# Draw zones
for zn,(yb,yt,hwb,hwt) in ZG.items():
    ax.add_patch(Polygon([(CX-hwt,yt),(CX+hwt,yt),(CX+hwb,yb),(CX-hwb,yb)],closed=True,facecolor=C[f'z{zn}f'],edgecolor=C[f'z{zn}e'],lw=0.7,zorder=2))

# ==============================
# Left-side numbering (01-06) — FINAL REDUCED FONT SIZE (12pt)
# ==============================
num_colors = [C['c1'], C['c2'], C['z3e'], C['z4e'], C['z5e'], C['z6e']]
for i, (zn, (yb,yt,_,_)) in enumerate(ZG.items(), start=1):
    y_mid = (yb + yt) / 2
    # Reduced font size to 12pt (smallest, cleanest look)
    T(ax, 0.45, y_mid, f"{i:02d}", num_colors[i-1], 12, ha='center', va='center', bold=True)

# ==============================
# Title
# ==============================
hl(ax,6.17,0.20,9.85,col=C['dk'],lw=1.4)
T(ax,4.90,6.09,'Urban Microclimate-Responsive Indoor Thermal Comfort in Subtropical Elderly Care Facilities',C['dk'],9.5,bold=True)
T(ax,4.90,5.96,'Guiyang Summer 2025 | 13 Facilities | Gradient Boosting + SHAP + NSGA-II',C['mu'],7.0)
hl(ax,5.88,0.20,9.85,col=C['lt'],lw=0.3)

# ==============================
# Z1
# ==============================
yb1,yt1=ZG[1][:2]; CHY=yt1-0.30; hw1=hw_at(CHY,1)
chips=[(CX-hw1+0.10,0.88,0.32,C['c1'],'Solar radiation','0.362'),
       (CX-hw1+1.06,0.72,0.26,C['c2'],'WWR','0.157'),
       (CX-hw1+1.84,0.64,0.22,C['c2'],'PM2.5','0.131'),
       (CX-hw1+2.54,0.56,0.18,C['c3'],'RH','0.096'),
       (CX-hw1+3.16,0.52,0.16,C['c3'],'CO2','0.091'),
       (CX-hw1+3.74,0.50,0.15,C['c4'],'TG','0.088')]
for xl,w,h,fc,l1,l2 in chips:
    rbox(ax,xl,CHY-h/2,w,h,fc=fc,r=0.04,z=6)
    T(ax,xl+w/2,CHY+h*0.13,l1,C['wh'],7.0,bold=True,z=7)
    T(ax,xl+w/2,CHY-h*0.26,l2,'#C8DCF0',6.0,z=7)

# Place main text on its own line
main_text_y = yb1+0.22
T(ax,CX,main_text_y,'Urban Microclimate · Guiyang · 5 Districts',C['z1d'],8.0,bold=True)

# Place "Ta · Wind · Noise" on the line BELOW the main text
pill_y_negl = yb1+0.10
pill(ax,CX, pill_y_negl, 1.80, 0.14, C['fa'],z=5)
T(ax,CX, pill_y_negl,'Ta · Wind · Noise (SHAP<0.05)',C['mu'],6.5)

# Z2
yb2,yt2=ZG[2][:2]; ym2=(yb2+yt2)/2
T(ax,CX,ym2+0.12,'Building Envelope · Window-to-Wall Ratio',C['z2d'],8.0,bold=True)
T(ax,CX,ym2-0.14,'WWR 0.10-0.73 (mean 0.36) · CAD-measured · 13 facilities',C['z2e'],7.0)

# Z3
yb3,yt3=ZG[3][:2]; ym3=(yb3+yt3)/2
T(ax,CX,ym3+0.16,'168-h Monitoring · n=234 Questionnaires (ASHRAE 55-2023)',C['z3d'],8.0,bold=True)
T(ax,CX,ym3-0.03,'Days 1,4,7 · 10:00 & 15:00 · 3 respondents per time-point',C['z3e'],7.0)
hw3p=hw_at(yb3+0.20,3)
for xc,lbl in [(CX-hw3p+0.74,'2,184 records'),(CX+0.22,'n=78 merged')]:
    pill(ax,xc,yb3+0.20,1.18,0.24,C['z3e'],z=5)
    T(ax,xc,yb3+0.20,lbl,C['wh'],7.0)

# Z4
yb4,yt4=ZG[4][:2]; ym4=(yb4+yt4)/2
T(ax,CX,ym4+0.14,'Gradient Boosting + LOO Cross-Validation',C['z4d'],8.0,bold=True)
T(ax,CX,ym4-0.12,'R2=0.3734 · RMSE=0.632 · MAE=0.487 · p=0.782',C['z4e'],7.0)

# Z5
yb5,yt5=ZG[5][:2]
T(ax,CX,yt5-0.20,'SHAP Interpretability (TreeExplainer)',C['z5d'],8.0,bold=True)
BAR_XL=CX-1.30; BAR_MAX=1.50; BAR_H=0.15; BAR_TOP=yt5-0.50; BAR_GAP=0.22
shap_rows=[(1.000,C['s1'],'Solar radiation','0.362'),(0.434,C['s2'],'WWR','0.157'),(0.362,C['s3'],'PM2.5','0.131'),(0.265,C['s4'],'Humidity RH','0.096')]
for i,(frac,fc,lbl,val) in enumerate(shap_rows):
    bw=frac*BAR_MAX; bcy=BAR_TOP-i*BAR_GAP
    rbox(ax,BAR_XL,bcy-BAR_H/2,bw,BAR_H,fc=fc,r=0.03,z=7)
    xr=BAR_XL + bw + 0.10
    T(ax,xr,bcy,f'{lbl}   {val}',fc,7.0,ha='left',bold=(frac==1.0))
T(ax,BAR_XL+0.1,BAR_TOP-4*BAR_GAP-0.04,'+ CO2 · TG · Ta · Noise · Wind',C['mu'],6.5,ha='left',italic=True)

# Z6
yb6,yt6=ZG[6][:2]; hw6t=hw_at(yt6-0.12,6)
pill(ax,CX,yt6-0.12,min(2*hw6t-0.20,3.60),0.20,C['z5e'],z=5)
T(ax,CX,yt6-0.12,'WWR x radiation · 27 C nonlinear threshold',C['wh'],6.5)
T(ax,CX,(yb6+yt6)/2+0.10,'NSGA-II Multi-Objective Optimisation',C['z6d'],8.0,bold=True)
T(ax,CX,(yb6+yt6)/2-0.14,'Min P(TSV>0.5) x Energy · WWR[0.10,0.73] · ACH[0.5,3.0] · Knee:WWR=0.43',C['z6e'],6.5)

# Bottom output
rbox(ax,0.20,OUT_YB,6.60,OUT_H,fc=C['z7f'],r=0.06,z=3)
for xc,txt in [(1.00,'WWR=0.43'),(2.36,'Discomfort ↓15%'),(3.68,'Energy ↓10%'),(5.00,'CO₂ ↓3.2t/yr')]:
    pill(ax,xc,OUT_YB+0.28,1.08,0.22,C['wh'],z=6)
    T(ax,xc,OUT_YB+0.28,txt,C['z7f'],7.0,bold=True,z=7)
T(ax,3.50,OUT_YB+0.07,'Evidence-based retrofit guidelines · Healthy China 2030','#F5B7B1',6.0,z=7)

# Right panel
rbox(ax,RP_X,OUT_YB,RP_W,RP_YT-OUT_YB,fc='#F8FAFB',ec=C['lt'],lw=0.5,r=0.08,z=2)
rbox(ax,RP_X,RP_YT-0.34,RP_W,0.34,fc=C['dk'],r=0.08,z=4)
RC=RP_X+RP_W/2; RXL=RP_X+0.13; RXR=RP_X+RP_W-0.10
T(ax,RC,RP_YT-0.17,'Data & Results Summary',C['wh'],8.0,bold=True)

def psec(y0,title,hc,rows):
    pill(ax,RC,y0-0.12,RP_W-0.22,0.22,hc,z=5)
    T(ax,RC,y0-0.12,title,C['wh'],7.5,bold=True)
    y=y0-0.34
    for lbl,val,vc in rows:
        T(ax,RXL,y,lbl,C['mu'],7.0,ha='left')
        T(ax,RXR,y,val,vc,7.0,ha='right',bold=True)
        y-=0.22
    hl(ax,y-0.04,RP_X+0.10,RP_X+RP_W-0.10,col=C['lt'],lw=0.3)
    return y-0.04

py=RP_YT-0.46
py=psec(py,'Study design',C['z3e'],[('Facilities','13',C['z3d']),('Monitoring','168 h',C['z3e']),('Questionnaires','n=234',C['z3e']),('Merged means','n=78',C['z3e'])])
py=psec(py-0.14,'Model performance',C['z4e'],[('LOO R2','0.3734',C['z4d']),('RMSE','0.632',C['z4e']),('MAE','0.487',C['z4e']),('S-W p','0.782',C['z4e'])])
py=psec(py-0.14,'SHAP features',C['z5e'],[('Solar rad.','0.362',C['s1']),('WWR','0.157',C['s2']),('PM2.5','0.131',C['s3']),('RH','0.096',C['s4'])])
psec(py-0.14,'Design output',C['z6e'],[('Optimal WWR','0.43',C['z6d']),('Discomfort','↓15%',C['z6e']),('Energy','↓10%',C['z6e']),('CO2','3.2t/yr',C['z6e'])])

# Footer
hl(ax,0.14,0.20,9.85,col=C['lt'],lw=0.3)
T(ax,0.25,0.07,'13 facilities · Guiyang, Guizhou · July-August 2025',C['mu'],6.0,ha='left')
T(ax,9.70,0.07,'Zenodo DOI: 10.5281/zenodo.19499335',C['mu'],6.0,ha='right')

for fmt in ['png','tiff']:
    fig.savefig(f'/content/figures/fig1_final_final.png',dpi=600,bbox_inches='tight',facecolor='white')
plt.close()
print('✅ FIGURE 1 — FINAL FINAL VERSION GENERATED!')

from google.colab import files
files.download('/content/figures/fig1_final_final.png')

✅ FIGURE 1 — FINAL FINAL VERSION GENERATED!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>